In [142]:
import gurobipy as gp
from gurobipy import GRB
import pandas as pd
import math

## Data loading

In [143]:
a1 = pd.read_csv("data/Area_1.txt", sep="\t")
a2 = pd.read_csv("data/Area_2.txt", sep="\t")
a3 = pd.read_csv("data/Area_3.txt", sep="\t")

all_cities = pd.concat([a1, a2, a3], axis=0).set_index("City")

## Helpers

In [144]:
def calculate_distacnce(all_cities, W, S) -> pd.DataFrame:
    D = pd.DataFrame(index=W, columns=S, dtype=float)
    for i in W:
        i_x = all_cities.loc[i, :]["x"]
        i_y = all_cities.loc[i, :]["y"]

        for s in S:
            s_x = all_cities.loc[s, :]["x"]
            s_y = all_cities.loc[s, :]["y"]

            D.loc[i, s] = math.sqrt((i_x - s_x)**2 + (i_y - s_y)**2)

    return D

def calculate_U(D, W, S) -> pd.DataFrame:
    U = pd.DataFrame(index=W, columns=S, dtype=float)

    for i in W:
        for s in S:
            U.loc[i, s] = max(
                0, min(200, 2 * math.pi * min(35, 70 - D.loc[i, s]))
            )

    return U

## Sets and parameters

In [145]:
W = [
    "Łódź",
    "Konin",
    "Sieradz",
    "Częstochowa",
    "Bytom",
    "Sosnowiec",
    "Katowice",
    "Wodzisław Śląski",
    "Poznań",
    "Zielona Góra",
    "Wrocław",
    "Wałbrzych"
]

S = ["Warszawa", "Kielce", "Opole"]

E = [(i, s) for i in W for s in S]

D = calculate_distacnce(all_cities, W, S)

p = 1000 #EUR

C = pd.Series({
    "Warszawa": 800,
    "Kielce": 1200,
    "Opole": 400
})

q = (all_cities[all_cities.index.isin(W)])["TB/s"].astype(int)

U = calculate_U(D, W, S)

## Model implementation

In [146]:
model = gp.Model("Copper_Wire")
model.setParam("OutputFlag", 0)

# Decision variable
x = model.addVars(
    W, S,
    lb=0,
    vtype=GRB.CONTINUOUS,
    name="x"
)

# Objective function
model.setObjective(
    gp.quicksum(
        p * D.loc[i, s] * x[i, s]
        for i in W
        for s in S
    ),
    GRB.MINIMIZE
)

# Constraint 1
model.addConstrs((
        x[i, s] <= U.loc[i, s]
        for i in W
        for s in S
    ),
    name="Cable_capacity"
)

# Constraint 2
model.addConstrs((
        gp.quicksum(x[i, s] for s in S) == q[i]
        for i in W
    ),
    name="Demand"
)

# Constraint 3
model.addConstrs(
    (
        gp.quicksum(x[i, s] for i in W) <= C[s]
        for s in S
    ),
    name="Station_capacity"
)

# Constraint 4
model.addConstrs((
        x["Łódź", s] <= 0.4 * q["Łódź"]
        for s in S
    ),
    name="Lodz_limit"
)

{'Warszawa': <gurobi.Constr *Awaiting Model Update*>,
 'Kielce': <gurobi.Constr *Awaiting Model Update*>,
 'Opole': <gurobi.Constr *Awaiting Model Update*>}

## Model optimization

In [149]:
model.optimize()

if model.Status == GRB.OPTIMAL:
    print(f"Optimal cost: {model.ObjVal:,.2f} EUR")

    results = []

    for i in W:
        for s in S:
            if x[i, s].X > 1e-6:
                results.append({
                    "City": i,
                    "Switch": s,
                    "Capacity": x[i, s].X
                })

    solution_df = pd.DataFrame(
        results,
        columns=["City", "Switch", "Capacity"]
    )

    display(solution_df)

solution_df.to_csv("copper_wire_sol.csv")

Optimal cost: 66,696,434.43 EUR


,City,Switch,Capacity
0,Łódź,Warszawa,126.800000
1,Łódź,Kielce,126.800000
2,Łódź,Opole,63.400000
3,Konin,Warszawa,200.000000
4,Konin,Kielce,36.000000
5,Sieradz,Warszawa,118.864243
6,Sieradz,Kielce,59.135757
7,Częstochowa,Warszawa,78.000000
8,Częstochowa,Kielce,200.000000
9,Bytom,Kielce,163.000000


In [152]:
solution_df[solution_df["Switch"] == "Kielce"]["Capacity"].sum()

np.float64(1200.0)

In [153]:
from modules.check_solution_of_copper_wire import check_solution

check_result = check_solution("copper_wire_sol.csv")
print(check_result)


Copper-wire checker: PASS
[PASS] Input: Read 22 rows; expected 12 cities outside the main line.
[PASS] Only permitted city-station links: Unexpected links: none.
[PASS] No parallel cables: Repeated city-station pairs: none.
[PASS] Nonnegative cable capacities: Negative entries: none.
[PASS] 200 TB/s maximum per cable: Violations: none.
[PASS] Distance-dependent capacity: Violations (city, station, actual, max): none.
[PASS] Demand fully met: Violations (city, supplied, required): none.
[PASS] Every remaining city connected: Unconnected cities: none.
[PASS] Switching-station capacity: Overloaded (station, use, max): none.
[PASS] Łódź: maximum 40% per cable: Maximum 126.800000 TB/s per cable; violations: none.
[PASS] Cost independently recalculated: Cost computed as 1000 * Euclidean distance * installed TB/s; no reported cost supplied.

Recalculated total cost: 66,696,434.43 EUR
Switching-station utilization:
  Warszawa: 731.000000 / 800 TB/s (91.38%)
  Kielce: 1200.000000 / 1200 TB/s (